# Probe runtime Colab cho đồ án Llama sinh mã

Repo: https://github.com/datkony/llama-code-generation.git

**Nguyên tắc quan trọng:** khi kernel là Colab, cell chạy trên máy ảo của Google, không phải máy bạn. File `F:\gitproject\llama-code-generation` không có ở đó, nên cell 2 phải copy repo vào máy ảo. Đây không phải repo thứ hai.

Extension `google.colab` trên VS Code **không đọc được `google.colab.userdata`** (báo TimeoutException). Vì vậy notebook dùng `getpass` để nhập token trực tiếp — token không được ghi vào file notebook và không in ra màn hình.

Repo hiện đang **private** nên `git clone` ẩn danh sẽ hỏi username. Cell 2 thử clone public trước; nếu thất bại sẽ hỏi **GitHub PAT** (scope `repo`). Cách khác: đổi repo sang public trong GitHub Settings rồi chạy lại cell 2, không cần PAT.

HF token: truy cập https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct → chấp nhận license → tạo token tại https://huggingface.co/settings/tokens. Không dán token vào code/notebook; chỉ nhập khi cell 4 hỏi.

Chạy lần lượt cell 1 → 2 → 3 → 4 → 5. Nếu chưa có token, cell 5 chỉ chạy `--offline` để kiểm tra pipeline — số liệu offline KHÔNG được đưa vào báo cáo.

In [ ]:
# Colab có sẵn diffusers 0.42.1 đòi huggingface-hub>=1.32, xung đột với transformers 4.x.
# Đồ án không dùng diffusers → gỡ bỏ để pip không báo dependency conflict.
!pip -q uninstall -y diffusers
!pip -q install "transformers>=4.49,<5" "accelerate>=1.3,<2" "huggingface_hub>=0.30,<1" "bitsandbytes>=0.45,<1" gradio

import torch, transformers, huggingface_hub
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("transformers", transformers.__version__, "| hub", huggingface_hub.__version__)

In [ ]:
import os, subprocess
from getpass import getpass

REPO_URL = "https://github.com/datkony/llama-code-generation.git"
TARGET = "llama-code-generation"
env = os.environ.copy()
env["GIT_TERMINAL_PROMPT"] = "0"  # không để git treo chờ hỏi username

if os.path.isdir(os.path.join(TARGET, ".git")):
    print("Repo đã có trong Colab → bỏ qua clone.")
else:
    try:
        subprocess.run(["git", "clone", "-q", REPO_URL, TARGET], check=True, timeout=180, env=env)
        print("Clone public thành công.")
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
        print("Clone public thất bại (repo đang private).")
        token = getpass("GitHub PAT (scope repo), hoặc Enter nếu đã đổi repo sang public: ").strip()
        if not token:
            raise RuntimeError("Cần GitHub PAT hoặc đổi repo thành public rồi chạy lại cell này.")
        auth_url = REPO_URL.replace("https://", f"https://{token}@")
        try:
            subprocess.run(["git", "clone", "-q", auth_url, TARGET], check=True, timeout=180, env=env)
            print("Clone bằng GitHub PAT thành công.")
            subprocess.run(["git", "-C", TARGET, "remote", "set-url", "origin", REPO_URL], check=True, env=env)
            print("Đã xóa PAT khỏi .git/config.")
        finally:
            token = ""
            auth_url = ""

os.chdir(TARGET)
print("Working dir:", os.getcwd())

In [ ]:
!python scripts/check_runtime.py

In [ ]:
import os
from getpass import getpass

token = os.environ.get("HF_TOKEN", "")
if not token:
    token = getpass("HF token (Enter nếu chưa có): ").strip()
if token:
    os.environ["HF_TOKEN"] = token
    print("HF_TOKEN present: True")
else:
    print("HF_TOKEN present: False → bỏ qua smoke endpoint.")
token = ""  # xóa biến Python; env vẫn giữ để tiến trình con dùng

if os.environ.get("HF_TOKEN"):
    get_ipython().system("python scripts/smoke_test.py --backend endpoint")
else:
    print("Cần HF token + chấp nhận license Llama 3.1 để chạy model thật.")

In [ ]:
import os

if os.environ.get("HF_TOKEN"):
    get_ipython().system("python scripts/run_benchmark.py --variants A,B")
else:
    print("Chưa có HF_TOKEN → chỉ chạy --offline để validate pipeline. KHÔNG báo cáo số liệu này.")
    get_ipython().system("python scripts/run_benchmark.py --offline --variants A,B")
get_ipython().system("python -m unittest discover -s tests -v")